# Exercícios - Lidando com valores ausentes

### Carregar as bibliotecas *necessárias*

In [ ]:
import pandas as pd
# Para o exercício 9
from sklearn.impute import KNNImputer

### Carregar o dataset (`EstudoSalarios.csv`)

In [ ]:
dados = pd.read_csv("EstudoSalarios.csv")

In [ ]:
dados.shape

(5000, 6)

## Para cada um dos exercícios abaixo, recarregue o dataset original

### 1. Mostrar os valores ausentes de cada feature



In [ ]:
dados.isna().sum()

,0
Unnamed: 0,0
idade,397
salario,0
sexo,178
cidade,93
nivel,0


In [ ]:
dados.dtypes

,0
Unnamed: 0,int64
idade,float64
salario,float64
sexo,object
cidade,object
nivel,object


### 2. Imputar valores ausentes com a média Preencha os valores ausentes na coluna `idade` com a média dessa coluna.

In [ ]:
media_idade = dados["idade"].mean()
dados2 = dados.fillna({"idade": media_idade})

In [ ]:
dados2.isna().sum()

,0
Unnamed: 0,0
idade,0
salario,0
sexo,178
cidade,93
nivel,0


### 3. Imputar valores ausentes com a mediana Preencha os valores ausentes na coluna `salario` com a mediana dessa coluna.

In [ ]:
salario_mediana = dados['salario'].median()
dados3 = dados.fillna({'salario': salario_mediana})

### 4. Criar uma coluna indicadora para os valores ausentes Crie uma nova coluna chamada 'idade_ausente' que indique se o valor na coluna `idade` está ausente (1 para ausente, 0 para presente).

In [ ]:
dados4 = dados.copy()
dados4['idade_ausente'] = dados['idade'].isna().astype(int)

### 5. Remover linhas com valores ausentes em uma coluna específica Remova todas as linhas onde a coluna `sexo` está com valores ausentes.

In [ ]:
dados5 = dados.dropna(subset=['sexo'])

In [ ]:
dados5.shape

(4822, 6)

### 6. Preencher valores ausentes com o valor mais frequente Preencha os valores ausentes na coluna `cidade` com o valor mais frequente (moda).

In [ ]:
cidade_moda = dados['cidade'].mode()[0]   # Moda cria um dataframe para empates
cidade_moda

'Rio de Janeiro'

In [ ]:
dados6 = dados.fillna({'cidade': cidade_moda})

### 7. Imputar valores ausentes com base em agrupamento Preencha os valores ausentes na coluna 'idade' com a média da idade dos funcionários de acordo com o nível (`nivel`).

In [ ]:
idade_por_nivel = dados.groupby('nivel')['idade'].mean()

In [ ]:
dados[dados['idade'].isna()]['nivel'].map(idade_por_nivel)

,nivel
22,43.482759
47,43.482759
49,43.430740
69,43.430740
83,43.482759
...,...
4959,43.430740
4969,43.482759
4978,43.430740
4996,43.482759


In [ ]:
dados[dados['idade'].isna()]

,Unnamed: 0,idade,salario,sexo,cidade,nivel,idade_ausente
22,22,NaN,48032.264739,Feminino,Curitiba,diretoria,1
47,47,NaN,32850.907397,Feminino,Curitiba,diretoria,1
49,49,NaN,18587.182842,Feminino,São Paulo,gerencia,1
69,69,NaN,14566.538776,Feminino,São Paulo,gerencia,1
83,83,NaN,30135.019918,Masculino,Curitiba,diretoria,1
...,...,...,...,...,...,...,...
4959,4959,NaN,13566.378582,Feminino,São Paulo,gerencia,1
4969,4969,NaN,36559.721998,Feminino,Rio de Janeiro,diretoria,1
4978,4978,NaN,19476.654724,Feminino,Curitiba,gerencia,1
4996,4996,NaN,30786.385388,Feminino,Rio de Janeiro,diretoria,1


In [ ]:
dados7 = dados.copy()
dados7[dados['idade'].isna()]['idade'] = dados[dados['idade'].isna()]['nivel'].map(idade_por_nivel)

<ipython-input-21-63ea3509ab7e>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dados7[dados['idade'].isna()]['idade'] = dados[dados['idade'].isna()]['nivel'].map(idade_por_nivel)


### 8. Remover colunas com muitos valores ausentes Remova qualquer coluna que tenha mais de 5% de valores ausentes.

In [ ]:
porcentagens = dados.isna().sum()/len(dados)
porcentagens

,0
Unnamed: 0,0.0000
idade,0.0794
salario,0.0000
sexo,0.0356
cidade,0.0186
nivel,0.0000


In [ ]:
porcentagens > 0.05

,0
Unnamed: 0,False
idade,True
salario,False
sexo,False
cidade,False
nivel,False
idade_ausente,False


In [ ]:
dados8 = dados.loc[:,porcentagens <= 0.05]

### 9. Imputação baseada em KNN Use o KNNImputer para preencher os valores ausentes na coluna `salario`.

In [ ]:
imputer = KNNImputer(n_neighbors=2)

numer = [var for var, tipo in zip(dados.columns, dados.dtypes) if tipo in ["float64", "int64"]]
outro = [var for var, tipo in zip(dados.columns, dados.dtypes) if tipo not in ["float64", "int64"]]

dados_num = dados[numer]

dados9 = imputer.fit_transform(dados_num)
dados9 = pd.DataFrame(dados9, columns = dados_num.columns, index=dados.index)

dados9 = dados9.join(dados[outro])
dados9.head()

,Unnamed: 0,idade,salario,sexo,cidade,nivel
0,0.0,56.0,2327.322752,Masculino,Curitiba,analista
1,1.0,69.0,3561.061535,Feminino,São Paulo,analista
2,2.0,46.0,17285.124778,Masculino,São Paulo,gerencia
3,3.0,32.0,3739.552653,Masculino,Rio de Janeiro,analista
4,4.0,60.0,10103.896574,Masculino,NaN,gerencia


### 10. Remover linhas com mais de 2 valores ausentes Remova as linhas do DataFrame que tenham mais de 2 valores ausentes.

In [ ]:
ausentes = dados.isna().sum(axis=1)

dados10 = dados[ausentes<=2]